In [5]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.medgan.models import MEDGAN
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = Path.cwd().parents[1]

raw_file = ROOT / "datasets" / "car.csv"
processed_file = ROOT / "preprocessed_data" / "car.csv"
artifact_dir = ROOT / "artifacts"

print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file)
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=MEDGAN(
        ae_pretrain_epochs=10,
        gan_epochs=10
    )
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="car",
    artifact_store=store,
    model_name="medgan",
)

print(results)

INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Training MedGAN Model
INFO:katabatic.models.medgan.models:================================================================================
INFO:katabatic.models.medgan.models:Loaded training data: (1382, 6)
INFO:katabatic.models.medgan.models:Categorical columns: ['0', '1', '2', '3', '4', '5', '6']
INFO:katabatic.models.medgan.models:Continuous columns: []
INFO:katabatic.models.medgan.models:Data normalized to [0, 1] range
INFO:katabatic.models.medgan.models:Original range: [0.00, 3.00]
INFO:katabatic.models.medgan.models:Normalized range: [0.00, 1.00]
INFO:katabatic.models.medgan.models:
Phase 1: Pretraining Autoencoder for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: AE Loss = 0.661285


Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\car.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\car.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\car.csv
Loaded data with shape: (1728, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car/split-20260807-142308


INFO:katabatic.models.medgan.models:Epoch 10/10: AE Loss = 0.380704
INFO:katabatic.models.medgan.models:
Phase 2: Training GAN for 10 epochs...
INFO:katabatic.models.medgan.models:Epoch 1/10: D Loss = 1.375065, G Loss = 0.712238
INFO:katabatic.models.medgan.models:Epoch 10/10: D Loss = 1.192887, G Loss = 1.209715
INFO:katabatic.models.medgan.models:
Generating 1382 synthetic samples...
INFO:katabatic.models.medgan.models:Adding one existing training sample for missing target classes...
INFO:katabatic.models.medgan.models:
Synthetic data saved to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\models\medgan_car_train-20260807-142308\synthetic
INFO:katabatic.models.medgan.models:Training complete!


{'message': 'Train test split pipeline executed successfully.', 'dataset_ref': DatasetRef(dataset_name='car', dataset_version='split-20260807-142308'), 'model_ref': ModelRef(model_name='medgan', dataset_name='car', dataset_version='split-20260807-142308', train_run_id='train-20260807-142308'), 'evaluation_refs': []}


In [6]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "car"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = train_df.columns[-1]

categorical_cols = train_df.columns[:-1].tolist()
continuous_cols = []

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Synthetic type:", type(synthetic_df))
print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\car\split-20260807-142308
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Synthetic sample:
       0     1  2  3      4     5      6
0  vhigh   low  2  2  small  high   good
1  vhigh   low  2  2  small  high  unacc
2  vhigh  high  2  2  small  high   good
3  vhigh  high  2  2  small  high   good
4  vhigh  high  2  2  small  high  unacc

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.4386

Categorical JSD (lower = better)  ->  score: 0.4386
  0                              JSD = 0.6114
  1                              JSD = 0.4679
  2                              JSD = 0.5955
  3                              JSD = 0.5667
  4                              JSD = 0.5667
  5                              JSD = 0.5600
  avg                            JSD = 0.5614

Running utility evaluation...


c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780:


=== Utility Evaluation ===
Overall utility score: 0.7485

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.5763       0.6884       0.1121
LR           f1         0.5193       0.6097       0.0904
DT           accuracy   0.6994       0.9734       0.274
DT           f1         0.5757       0.9730       0.3973
RF           accuracy   0.6994       0.9671       0.2677
RF           f1         0.5757       0.9668       0.3911
LinearSVM    accuracy   0.5543       0.7023       0.148
LinearSVM    f1         0.5103       0.6239       0.1136
MLP          accuracy   0.6624       0.9717       0.3093
MLP          f1         0.5597       0.9717       0.412

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.3725

Category Coverage (% of real categories in synth)
  0                              25.0%
  1                              50.0%
  2                              5